In [1]:
import os
os.chdir("../")
%pwd

'/Users/ayush/Desktop/Coding-practice/CampusX/Projects/End_to_end-Kidney-Disease-Classification'

In [2]:
import os
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/ayush99coding/Kidney-Disease-Classification.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "ayush99coding"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "c19d654ab3862dab7fea46d4a763e2bc2769ec9e"

In [3]:
import tensorflow as tf
model = tf.keras.models.load_model("artifacts/training/model.h5")

2026-07-31 02:07:54.934147: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [5]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney-ct-scan-image",
            mlflow_uri="https://dagshub.com/ayush99coding/Kidney-Disease-Classification.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [6]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):

        mlflow.set_tracking_uri(self.config.mlflow_uri)

        with mlflow.start_run():

            mlflow.log_params(self.config.all_params)

            mlflow.log_metrics({
                "loss": self.score[0],
                "accuracy": self.score[1]
            })

            mlflow.keras.log_model(
                self.model,
                artifact_path="model"
            )

/Users/ayush/Desktop/Coding-practice/CampusX/Projects/End_to_end-Kidney-Disease-Classification/venv/lib/python3.10/site-packages/mlflow/utils/requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


In [7]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

[2026-07-31 02:08:37,592: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-07-31 02:08:37,604: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-31 02:08:37,607: INFO: common: created directory at: artifacts]
Found 139 images belonging to 2 classes.
9/9 [==============================] - 59s 7s/step - loss: 1.6263 - accuracy: 0.6259
[2026-07-31 02:09:38,185: INFO: common: json file saved at: scores.json]


2026/07/31 02:09:42 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: /var/folders/50/y9cm8_l96597fg52kqxk6ykw0000gn/T/tmpk_k8_ash/model/data/model/assets
[2026-07-31 02:09:52,604: INFO: builder_impl: Assets written to: /var/folders/50/y9cm8_l96597fg52kqxk6ykw0000gn/T/tmpk_k8_ash/model/data/model/assets]


/Users/ayush/Desktop/Coding-practice/CampusX/Projects/End_to_end-Kidney-Disease-Classification/venv/lib/python3.10/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
2026/07/31 02:11:06 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
2026/07/31 02:11:07 INFO mlflow.tracking._tracking_service.client: 🏃 View run selective-turtle-738 at: https://dagshub.com/ayush99coding/Kidney-Disease-Classification.mlflow/#/experiments/0/runs/d65c9f14dd254bd48a8821aa89a85a8c.
2026/07/31 02:11:07 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://dagshub.com/ayush99coding/Kidney-Disease-Classification.mlflow/#/exp